# Plaque-Associated Microglia: Gene Expression & GRN Spatial Analysis
---
**Analysis Pipeline**
1. Environment & Data Loading
2. Part A: Epigenetic gene expression vs plaque distance
3. Part B: GRN (regulon) activity vs plaque distance — linear regression + visualization

## 1. Environment & Data Loading

In [ ]:
# ==========================================
# 1.1 Libraries (all at once)
# ==========================================
library(Signac)
library(Seurat)
library(dplyr)
library(tidyr)
library(ggplot2)
library(stringr)
library(mgcv)
library(stats)
library(cowplot)
library(patchwork)
library(ggpubr)

print("Libraries loaded.")

In [ ]:
# ==========================================
# 1.2 Read spatial metadata & clean barcode
# ==========================================
spatial <- read.csv('/home1/yzhang/Project_SN_Multiome/Analysis/Spatial_integration/sn_plaqueD_metadata.csv')
spatial$Clean_Barcode <- gsub("@.*$", "", spatial$X)
spatial <- spatial %>%
  select(X, Clean_Barcode, sample_id, min_center_dist) %>%
  mutate(dist_um = min_center_dist / 2)

print(paste("Spatial:", nrow(spatial), "spots"))
head(spatial, 3)

In [ ]:
# ==========================================
# 1.3 Read multiome Seurat object & extract meta
# ==========================================
combined_full <- readRDS('/public/home/yzhang/zy_data/Project_AD_st/Multiome/Raw_data_0423_full/QC_0512/Combine/Combine_RNA_ATAC_anno_rnafromatac_260219.rds')
DefaultAssay(combined_full) <- "RNA"

meta <- combined_full@meta.data %>%
  select(cell_id, genotype, month, brain, class_id_label, Final_subclass_name)

print(paste("Combined:", ncol(combined_full), "cells"))
combined_full

In [ ]:
# ==========================================
# 1.4 Read GRN data: AUC matrix, DGRN, RSS
# ==========================================
# AUC matrix — keep only +/+ regulons
AUC_gene <- read.csv('/home1/yzhang/Project_SN_Multiome/Analysis/Scenic+/Version_2_mouse/gene_AUC_matrix.csv', check.names = FALSE)
cell_col_name <- colnames(AUC_gene)[1]
plus_cols <- colnames(AUC_gene)[grepl("\\+/\\+", colnames(AUC_gene))]
AUC_gene <- AUC_gene[, c(cell_col_name, plus_cols)]

# Fix cell barcodes: "barcode-sample" → "sample_barcode"
AUC_gene[[cell_col_name]] <- gsub("^([^-]+)-(.*)$", "\\2_\\1", AUC_gene[[cell_col_name]])

# Differential GRN & RSS
DGRN <- read.csv('/home1/yzhang/Project_SN_Multiome/Analysis/Scenic+/Version_2_mouse/Differential_GRN_0515.csv')
RSS <- read.csv('/home1/yzhang/Project_SN_Multiome/Analysis/Scenic+/Version_2_mouse/RSS/RSS_class_id_label.csv', row.names = 1)
RSS <- as.data.frame(t(RSS))

print(paste("AUC matrix:", ncol(AUC_gene), "columns | DGRN:", nrow(DGRN), "rows | RSS:", ncol(RSS), "cell types"))

In [ ]:
# ==========================================
# 1.5 Merge spatial + AUC + meta
# ==========================================
spatial <- spatial %>%
  inner_join(AUC_gene, by = c("Clean_Barcode" = "Cell")) %>%
  inner_join(meta, by = c("Clean_Barcode" = "cell_id"))

print(paste("Merged spatial:", nrow(spatial), "spots ×", ncol(spatial), "columns"))

## 2. Part A: Gene Expression vs Plaque Distance
Epigenetic regulators (Tet2, Asxl1, Kmt2d, Atrx, Cbl) in microglia

In [ ]:
# ==========================================
# 2.1 Subset microglia & extract expression
# ==========================================
mg <- subset(combined_full, Final_subclass_name == '334 Microglia NN')

target_genes <- c("Tet2", "Asxl1", "Kmt2d", "Atrx", "Cbl")
expr_data <- FetchData(mg, vars = target_genes)
expr_data$cell_id <- rownames(expr_data)

print(paste("Microglia:", ncol(mg), "cells"))

In [ ]:
# ==========================================
# 2.2 Binned mean expression vs plaque distance
#     (replaces original cells 15-22 with cleaner floor() approach)
# ==========================================
bin_size <- 25
max_dist <- 200

df_binned <- spatial %>%
  mutate(dist_um = min_center_dist / 2) %>%
  filter(dist_um <= max_dist) %>%
  inner_join(expr_data, by = c("Clean_Barcode" = "cell_id")) %>%
  mutate(bin_center = floor(dist_um / bin_size) * bin_size + bin_size / 2) %>%
  pivot_longer(cols = all_of(target_genes), names_to = "Gene", values_to = "Expression") %>%
  group_by(bin_center, brain, Gene) %>%
  summarise(
    Mean_Expr = mean(Expression, na.rm = TRUE),
    SE_Expr   = sd(Expression, na.rm = TRUE) / sqrt(n()),
    Count     = n(),
    .groups   = "drop"
  ) %>%
  filter(Count >= 3)

head(df_binned)

In [ ]:
# ==========================================
# 2.3 Plot: gene expression gradient
# ==========================================
ggplot(df_binned, aes(x = bin_center, y = Mean_Expr, color = Gene)) +
  geom_line(linewidth = 1) +
  geom_point(size = 2) +
  geom_errorbar(aes(ymin = Mean_Expr - SE_Expr, ymax = Mean_Expr + SE_Expr),
                width = 5, alpha = 0.5) +
  facet_wrap(~ brain, scales = "free_y") +
  theme_bw(base_size = 12) +
  labs(
    title    = "Epigenetic regulator expression vs plaque distance",
    subtitle = paste0("Bin size: ", bin_size, " µm | microglia only"),
    x        = "Distance to plaque center (µm)",
    y        = "Mean normalized expression"
  ) +
  theme(
    strip.background = element_rect(fill = "#f2f2f2", color = "black"),
    strip.text       = element_text(face = "bold", size = 11),
    legend.position  = "bottom"
  )

## 3. Part B: GRN Activity vs Plaque Distance
Linear regression + heatmap + single-TF line plot

In [ ]:
# ==========================================
# 3.1 Filter & build TF mapping
# ==========================================
spatial_sub <- spatial %>% filter(min_center_dist < 200)

grn_cols <- grep("\\+/\\+", colnames(spatial_sub), value = TRUE)

# Pure TF name → complex GRN name dictionary
pure_tfs  <- gsub("_direct_\\+/\\+.*$", "", grn_cols)
tf2grn    <- setNames(grn_cols, pure_tfs)

cell_types <- unique(na.omit(spatial_sub$class_id_label))
months     <- unique(na.omit(spatial_sub$month))

print(paste("Cell types:", length(cell_types), "| Months:", paste(months, collapse = ", "),
            "| GRN columns:", length(grn_cols)))

In [ ]:
# ==========================================
# 3.2 Precompute valid GRNs per cell type
#     (DGRN significant + RSS Top20 union)
# ==========================================
sig_diff_grn <- DGRN %>% filter(p_val_adj < 0.05 & abs(log2FC) > 0.1)

valid_grns_by_ctype <- lapply(cell_types, function(ctype) {
  # a. Differentially significant GRNs
  diff_grns <- sig_diff_grn %>%
    filter(CellType == ctype) %>%
    pull(Regulon) %>% unique()
  if (length(diff_grns) > 0 && !any(grepl("\\+/\\+", diff_grns))) {
    diff_grns <- na.omit(unname(tf2grn[diff_grns]))
  }
  
  # b. RSS Top 20
  if (ctype %in% colnames(RSS)) {
    top20 <- na.omit(unname(tf2grn[
      rownames(RSS)[order(RSS[[ctype]], decreasing = TRUE)[1:20]]
    ]))
  } else {
    top20 <- character(0)
  }
  
  intersect(unique(c(diff_grns, top20)), colnames(spatial_sub))
})
names(valid_grns_by_ctype) <- cell_types

print("GRN counts per cell type:")
sapply(valid_grns_by_ctype, length)

In [ ]:
# ==========================================
# 3.3 Linear regression: GRN ~ distance
#     Triple loop: cell_type → month → GRN
# ==========================================
results_list <- list()

for (ctype in cell_types) {
  grns_to_test <- valid_grns_by_ctype[[ctype]]
  if (length(grns_to_test) == 0) next
  
  for (m in months) {
    sub_data <- spatial_sub %>% filter(class_id_label == ctype, month == m)
    if (nrow(sub_data) < 30) next
    
    for (grn in grns_to_test) {
      tmp_df <- na.omit(data.frame(
        activity = sub_data[[grn]],
        dist     = sub_data$min_center_dist
      ))
      if (nrow(tmp_df) < 30) next
      
      fit <- lm(activity ~ dist, data = tmp_df)
      s <- summary(fit)$coefficients
      
      if ("dist" %in% rownames(s)) {
        results_list[[paste(ctype, m, grn, sep = "_")]] <- data.frame(
          CellType     = ctype,
          Month        = m,
          GRN          = grn,
          Beta_Distance = s["dist", "Estimate"],
          Pval         = s["dist", "Pr(>|t|)"],
          N_Cells      = nrow(tmp_df),
          stringsAsFactors = FALSE
        )
      }
    }
  }
}

# Aggregate & FDR correction
grn_stats <- bind_rows(results_list)
if (nrow(grn_stats) > 0) {
  grn_stats$FDR <- p.adjust(grn_stats$Pval, method = "BH")
  grn_stats <- grn_stats %>% arrange(FDR)
}

print(paste("Total tests:", nrow(grn_stats), "| Sig (FDR<0.05):", sum(grn_stats$FDR < 0.05)))
print(head(grn_stats, 10))

In [ ]:
# ==========================================
# 3.4 Heatmap: GRN activity Z-score by distance bin
#     Glial cells → per brain region; Neurons → merged
# ==========================================
dist_breaks <- c(seq(0, 200, by = 40), Inf)
bin_labels  <- c(paste0(seq(0, 160, by = 40), "-", seq(40, 200, by = 40)), ">=200")

for (ctype in names(valid_grns_by_ctype)) {
  ctype_stats <- grn_stats %>% filter(CellType == ctype & Pval < 0.05)
  sig_grns <- unique(ctype_stats$GRN)
  if (length(sig_grns) == 0) next
  
  is_glial <- !grepl("Glut|Gaba", ctype, ignore.case = TRUE)
  regions  <- if (is_glial) unique(na.omit(spatial$brain[spatial$class_id_label == ctype])) else "All"
  
  for (reg in regions) {
    pd <- spatial %>% filter(class_id_label == ctype)
    if (reg != "All") pd <- pd %>% filter(brain == reg)
    if (nrow(pd) == 0) next
    
    pd <- pd %>%
      select(month, min_center_dist, all_of(sig_grns)) %>%
      mutate(Dist_Bin = cut(min_center_dist, breaks = dist_breaks,
                            labels = bin_labels, include.lowest = TRUE)) %>%
      drop_na(Dist_Bin) %>%
      pivot_longer(all_of(sig_grns), names_to = "GRN", values_to = "Activity") %>%
      group_by(month, Dist_Bin, GRN) %>%
      summarise(Mean_Activity = mean(Activity, na.rm = TRUE), .groups = "drop") %>%
      group_by(GRN, month) %>%
      mutate(Z_Score = as.numeric(scale(Mean_Activity))) %>%
      ungroup() %>%
      left_join(ctype_stats %>% select(Month, GRN, Pval, Beta_Distance),
                by = c("month" = "Month", "GRN" = "GRN")) %>%
      mutate(
        Sig_Star = case_when(
          is.na(Pval) ~ "", Pval < 0.001 ~ "***",
          Pval < 0.01 ~ "**", Pval < 0.05 ~ "*", TRUE ~ ""
        ),
        Label = ifelse(Sig_Star != "", paste0(Sig_Star, "\n", signif(Beta_Distance, 2)), "")
      )
    
    first_bin <- levels(pd$Dist_Bin)[1]
    ann <- pd %>% filter(Dist_Bin == first_bin)
    
    title_str <- if (reg == "All") paste("GRN Activity in", ctype)
                else paste("GRN Activity in", ctype, "(", reg, ")")
    
    p <- ggplot(pd, aes(x = Dist_Bin, y = GRN, fill = Z_Score)) +
      geom_tile(color = "white", linewidth = 0.2) +
      geom_text(data = ann, aes(label = Label),
                color = "black", size = 2.5, fontface = "bold", hjust = 0, nudge_x = -0.4) +
      scale_fill_gradient2(low = "#313695", mid = "white", high = "#a50026", midpoint = 0) +
      scale_y_discrete(labels = function(x) gsub("_direct_\\+/\\+.*", "", x)) +
      facet_wrap(~ month, ncol = length(unique(pd$month))) +
      labs(title = title_str, x = "Distance (µm)", y = "GRN", fill = "Z-Score") +
      theme_minimal() +
      theme(
        axis.text.x = element_text(angle = 45, hjust = 1, size = 8),
        axis.text.y = element_text(size = 8),
        panel.grid   = element_blank(),
        strip.background = element_rect(fill = "grey90", color = NA)
      )
    print(p)
  }
}

In [ ]:
# ==========================================
# 3.5 Single-TF line plot example: Spi1 in Immune cells
#     Facet labels show β ± significance per month
# ==========================================
target_ctype <- "34 Immune"
target_tf    <- "Ebf1"

ctype_stats <- grn_stats %>% filter(CellType == target_ctype)
sig_grns <- ctype_stats %>%
  filter(grepl(paste0("^", target_tf, "_"), GRN)) %>%
  pull(GRN) %>% unique()

if (length(sig_grns) > 0) {
  plot_data <- spatial %>%
    filter(class_id_label == target_ctype) %>%
    select(month, min_center_dist, all_of(sig_grns)) %>%
    mutate(Dist_Bin = cut(min_center_dist, breaks = dist_breaks,
                          labels = bin_labels, include.lowest = TRUE)) %>%
    drop_na(Dist_Bin) %>%
    pivot_longer(all_of(sig_grns), names_to = "GRN", values_to = "Activity") %>%
    group_by(month, Dist_Bin, GRN) %>%
    summarise(Mean_Activity = mean(Activity, na.rm = TRUE), .groups = "drop") %>%
    group_by(GRN, month) %>%
    mutate(Z_Score = as.numeric(scale(Mean_Activity))) %>%
    ungroup() %>%
    left_join(ctype_stats %>% select(Month, GRN, Pval, Beta_Distance),
              by = c("month" = "Month", "GRN" = "GRN")) %>%
    mutate(
      Clean_GRN = gsub("_direct_\\+/\\+.*$", "", GRN),
      Sig_Star  = case_when(
        is.na(Pval) ~ "", Pval < 0.001 ~ "***",
        Pval < 0.01 ~ "**", Pval < 0.05 ~ "*", TRUE ~ "ns"
      ),
      Facet_Label = paste0(month, "M | β=", signif(Beta_Distance, 2), Sig_Star)
    )
  
  ggplot(plot_data, aes(x = Dist_Bin, y = Z_Score,
                        color = Clean_GRN, group = Clean_GRN)) +
    geom_line(linewidth = 1.2) +
    geom_point(size = 2.5) +
    geom_hline(yintercept = 0, linetype = "dashed", color = "grey50") +
    facet_wrap(~ Facet_Label, ncol = length(unique(plot_data$month))) +
    scale_color_manual(values = "#d73027") +
    labs(
      title = paste("GRN Trajectory:", target_tf, "in", target_ctype),
      x = "Distance to plaque (µm)", y = "Activity (Z-Score)"
    ) +
    theme_bw() +
    theme(
      axis.text.x = element_text(angle = 45, hjust = 1, size = 9),
      strip.background = element_rect(fill = "#E5E7EB", color = "black"),
      legend.position  = "bottom"
    )
}